In [0]:
%sql
CREATE OR REPLACE TABLE tnelections.form20_bronze._scratch_ocr_test AS
SELECT ac_no, ai_parse_document(content, map('version', '2.0')) AS parsed
FROM tnelections.form20_bronze.pdf_files
WHERE year = 2026 AND ac_no = 154;

In [0]:
%sql
SELECT parsed:error_status FROM tnelections.form20_bronze._scratch_ocr_test;

In [0]:
%sql
SELECT
  el.value:bbox[0]:page_id::int AS page,
  el.value:type::string         AS type,
  el.value:content::string      AS content
FROM tnelections.form20_bronze._scratch_ocr_test,
  LATERAL variant_explode(parsed:document:elements) AS el
ORDER BY page
LIMIT 20;

In [0]:
%sql
SELECT
  el.value:bbox[0]:page_id::int                 AS page,
  COUNT_IF(el.value:type::string = 'table')     AS tables,
  MAX(LENGTH(el.value:content::string))         AS longest_element
FROM tnelections.form20_bronze._scratch_ocr_test,
  LATERAL variant_explode(parsed:document:elements) AS el
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
SELECT
  el.value:bbox[0]:page_id::int AS page,
  el.value:type::string         AS type,
  el.value:content::string      AS content
FROM tnelections.form20_bronze._scratch_ocr_test,
  LATERAL variant_explode(parsed:document:elements) AS el
WHERE el.value:bbox[0]:page_id::int >= 19
ORDER BY page;

## OCR test: ai_parse_document on AC154 (Panruti), scanned, 21 pages, some stored upside down

**Result: works on Free Edition**
- All 21 pages returned one table each, including upside-down pages
- All booth rows on pages 0–4 checked: vote columns sum exactly to total valid votes
- Table ends with a "Postal Ballot" row and a grand total row; both tally
- Rejected votes (248) come only from postal ballots

**Watch-outs**
- Header layout changes page to page; column numbers leak into the header on the last page
- Candidate names have OCR slips (SAFI → SAFL), so take names from ECI, not the PDF
- The printed column-number row (1–17) is reliable; use it to anchor columns

**Decision**
- Scanned PDFs (9) → ai_parse_document
- Text PDFs (225) → pdfplumber, with ai_parse_document as fallback when tallies fail